In [8]:
import torch
import torch.nn as nn
from torchvision import models
import loralib as lora
import pandas as pd
from torchvision.models import ResNet18_Weights
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import time
import csv
from torchvision.datasets import ImageFolder
from pathlib import Path


# Pretrained ResNet50 weights
weights = ResNet18_Weights.IMAGENET1K_V1

# Use ResNet preprocessing
transform = weights.transforms()

VARIANT_ROOT = Path(r"C:\Users\preet\Documents\UBC_MDS\DATA586\Project")

variants = ["blur_little", "blur_medium", "clean", "downsampled", "masked", "noise_rotation"]
# variants = ["clean","blur_medium"]
# get correct mapping
train_dataset = datasets.Food101(root="./data", split="train", download=True)
class_to_idx = train_dataset.class_to_idx

def get_loader(variant):
    dataset = ImageFolder(
        root=VARIANT_ROOT / variant,
        transform=transform
    )

    loader = DataLoader(
        dataset,
        batch_size=64,
        shuffle=False,
        num_workers=8,
        pin_memory=True
    )

    return loader


Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x00000284098EBAC0>
Traceback (most recent call last):
  File "C:\Users\preet\miniconda3\envs\yolo\lib\site-packages\torch\utils\data\dataloader.py", line 1604, in __del__
    self._shutdown_workers()
  File "C:\Users\preet\miniconda3\envs\yolo\lib\site-packages\torch\utils\data\dataloader.py", line 1562, in _shutdown_workers
    if self._persistent_workers or self._workers_status[worker_id]:
AttributeError: '_MultiProcessingDataLoaderIter' object has no attribute '_workers_status'


In [9]:

def build_linear_probe(weights, num_classes=101):
    model = models.resnet18(weights=weights)

    for param in model.parameters():
        param.requires_grad = False

    model.fc = nn.Linear(model.fc.in_features, num_classes)

    for param in model.fc.parameters():
        param.requires_grad = True

    return model

    
def make_adapter(channels, reduction=16):
    hidden = max(channels // reduction, 4)

    adapter = nn.Sequential(
        nn.Conv2d(channels, hidden, kernel_size=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(hidden, channels, kernel_size=1),
    )

    # start as identity
    nn.init.zeros_(adapter[2].weight)
    nn.init.zeros_(adapter[2].bias)

    return adapter


def add_adapters(model):
    for layer_name in ["layer1", "layer2", "layer3", "layer4"]:
        layer = getattr(model, layer_name)

        for block in layer:
            adapter = make_adapter(block.conv2.out_channels)

            original_forward = block.forward

            def new_forward(x, orig=original_forward, adapter=adapter):
                out = orig(x)
                return out + adapter(out)
            block.forward = new_forward
            block.adapter = adapter  
            
def build_adapter(weights, num_classes=101):
    model = models.resnet18(weights=weights)

    add_adapters(model)

    model.fc = nn.Linear(model.fc.in_features, num_classes)

    for param in model.parameters():
        param.requires_grad = False

    for module in model.modules():
        if hasattr(module, "adapter"):
            for p in module.adapter.parameters():
                p.requires_grad = True

    for param in model.fc.parameters():
        param.requires_grad = True

    return model

def build_batchnorm(weights, num_classes=101):
    model = models.resnet18(weights=weights)

    model.fc = nn.Linear(model.fc.in_features, num_classes)

    for param in model.parameters():
        param.requires_grad = False

    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            for param in module.parameters():
                param.requires_grad = True
            module.train()

    for param in model.fc.parameters():
        param.requires_grad = True

    return model


def apply_lora_to_resnet(model, rank=8, alpha=16):
    def to_int(x):
        return x if isinstance(x, int) else x[0]

    def get_module(model, name):
        parts = name.split(".")
        m = model
        for p in parts:
            m = m[int(p)] if p.isdigit() else getattr(m, p)
        return m

    def set_module(model, name, new_module):
        parts = name.split(".")
        parent = model
        for p in parts[:-1]:
            parent = parent[int(p)] if p.isdigit() else getattr(parent, p)

        last = parts[-1]
        if last.isdigit():
            parent[int(last)] = new_module
        else:
            setattr(parent, last, new_module)

    target_layers = [
        "layer1.0.conv1", "layer1.0.conv2",
        "layer1.1.conv1", "layer1.1.conv2",
        "layer2.0.conv1", "layer2.0.conv2",
        "layer2.1.conv1", "layer2.1.conv2",
        "layer3.0.conv1", "layer3.0.conv2",
        "layer3.1.conv1", "layer3.1.conv2",
        "layer4.0.conv1", "layer4.0.conv2",
        "layer4.1.conv1", "layer4.1.conv2",
    ]

    for name in target_layers:
        module = get_module(model, name)

        new_layer = lora.Conv2d(
            in_channels=module.in_channels,
            out_channels=module.out_channels,
            kernel_size=to_int(module.kernel_size),
            stride=to_int(module.stride),
            padding=to_int(module.padding),
            dilation=to_int(module.dilation),
            groups=module.groups,
            bias=(module.bias is not None),
            r=rank,
            lora_alpha=alpha,
        )

        new_layer.conv.weight.data.copy_(module.weight.data)
        if module.bias is not None:
            new_layer.conv.bias.data.copy_(module.bias.data)

        set_module(model, name, new_layer)

    return model
def build_resnet18_bitfit(weights):
    """
    BitFit for ResNet18:
    - load ImageNet pretrained weights
    - replace classifier for Food101
    - freeze everything
    - unfreeze only:
        1) all bias parameters
        2) final classifier head
    """
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, 101)

    # Freeze everything first
    for param in model.parameters():
        param.requires_grad = False

    # Unfreeze all bias parameters
    for name, param in model.named_parameters():
        if "bias" in name:
            param.requires_grad = True

    # Unfreeze full classifier head
    for param in model.fc.parameters():
        param.requires_grad = True

    return model





In [10]:

import os
from pathlib import Path

BASE_DIR = Path.home() / "Documents/UBC_MDS/DATA586/Project_final/training_scripts/ResNet18/models"

experiments = [
    {
        "name": "linear_probe",
        "builder": build_linear_probe,
        "ckpt": BASE_DIR / "linear_probe_ResNet_best.pth"
    },
    {
        "name": "batchnorm",
        "builder": build_batchnorm,
        "ckpt": BASE_DIR / "batchnorm_finetune_best.pth"
    },
    {
        "name": "adapter",
        "builder": build_adapter,
        "ckpt":  BASE_DIR / "adapter_finetune_ResNet18_best.pth"
    },
    {
        "name": "lora",
        "builder": None,
        "ckpt": BASE_DIR / "lora_full_checkpoint.pth"
    }, 
    {
        "name": "bitfit",
        "builder": build_resnet18_bitfit,
        "ckpt": BASE_DIR / "ResNet18_BitFit_best.pth"
    },
]






device = "cuda" if torch.cuda.is_available() else "cpu"

def evaluate(model, loader, device):
    model.eval()
    correct, total = 0, 0

    start_time = time.perf_counter()

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            # IMPORTANT for accurate GPU timing
            if device == "cuda":
                torch.cuda.synchronize()

            outputs = model(x)

            if device == "cuda":
                torch.cuda.synchronize()

            preds = outputs.argmax(1)
            correct += (preds == y).sum().item()
            total += y.size(0)

    total_time = time.perf_counter() - start_time

    accuracy = correct / total
    time_per_sample = total_time / total
    time_per_batch = total_time / len(loader)

    return accuracy, total_time, time_per_sample, time_per_batch

results = []

for variant in variants:
    print(f"\n=== Variant: {variant} ===")

    test_loader = get_loader(variant)

    for exp in experiments:
        print(f"Running: {exp['name']}")
    
        
    
        ckpt = torch.load(exp["ckpt"], map_location=device)
        
        # handle both cases (robust)
    
        if exp["name"] == "lora":
            model = models.resnet18(weights=weights)
            model = apply_lora_to_resnet(model, rank=8, alpha=16)
    
            model.fc = nn.Linear(model.fc.in_features, 101)
            # print(model.fc.weight.mean(), model.fc.weight.std())
            
            model.load_state_dict(ckpt["model_state_dict"])
            
            model = model.to(device)
            model.eval()
            missing, unexpected = model.load_state_dict(ckpt["model_state_dict"], strict=False)
            # print("Missing:", missing)
            # print("Unexpected:", unexpected)
    
            # print(ckpt.keys())
            # print(model.fc.weight.mean(), model.fc.weight.std())
            
            
    
        else:
            model = exp["builder"](weights).to(device)
            # Normal full model loading
            ckpt = torch.load(exp["ckpt"], map_location=device)
            state_dict = ckpt["model_state_dict"] if "model_state_dict" in ckpt else ckpt
            model.load_state_dict(state_dict)

        acc, total_time, t_per_sample, t_per_batch = evaluate(model, test_loader, device)

        results.append({
            "variant": variant,
            "model": exp["name"],
            "accuracy": acc,
            "total_inference_time_sec": total_time,
            "time_per_sample_sec": t_per_sample,
            "time_per_batch_sec": t_per_batch,
        })

df = pd.DataFrame(results)
df.to_csv("results_variants_resnet_v2.csv", index=False)

print(df)





=== Variant: blur_little ===
Running: linear_probe


C:\Users\preet\AppData\Local\Temp\ipykernel_40308\4028912466.py:84: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(exp["ckpt"], map_location=device)
C:\User

Running: batchnorm
Running: adapter
Running: lora
Running: bitfit

=== Variant: blur_medium ===
Running: linear_probe
Running: batchnorm
Running: adapter
Running: lora
Running: bitfit

=== Variant: clean ===
Running: linear_probe
Running: batchnorm
Running: adapter
Running: lora
Running: bitfit

=== Variant: downsampled ===
Running: linear_probe
Running: batchnorm
Running: adapter
Running: lora
Running: bitfit

=== Variant: masked ===
Running: linear_probe
Running: batchnorm
Running: adapter
Running: lora
Running: bitfit

=== Variant: noise_rotation ===
Running: linear_probe
Running: batchnorm
Running: adapter
Running: lora
Running: bitfit
           variant         model  accuracy  total_inference_time_sec  \
0      blur_little  linear_probe  0.465188                 83.404228   
1      blur_little     batchnorm  0.600911                 68.823235   
2      blur_little       adapter  0.621267                 70.808382   
3      blur_little          lora  0.500238                 66.03